## OPT-350M 파인튜닝- Lora미적용

### 🔍 GPT-2 vs OPT-350M 상세 비교표

| 항목             | GPT-2                                     | OPT-350M (facebook/opt-350m)                      |
|------------------|--------------------------------------------|---------------------------------------------------|
| **개발자**        | OpenAI                                     | Meta (Facebook AI Research)                       |
| **공개 연도**     | 2019                                       | 2022                                              |
| **파라미터 수**    | 약 **1.2억 (small)**~15억 (large)             | 약 **3.5억**                                       |
| **학습 데이터**   | 웹페이지, 위키 등 일반 텍스트 (WebText)       | Open Pretraining Dataset (RoBERTa 등 기반)        |
| **학습 목적**     | 일반 언어 생성                              | 고효율, GPT 스타일 대안 (특히 학계/산업용 목적)     |
| **아키텍처**      | Decoder-only Transformer                  | GPT-2 기반 Decoder-only 구조 (단순하고 효율적)     |
| **라이선스**      | 오픈 (MIT 라이선스 아님)                     | 오픈 (Apache 2.0 라이선스)                         |
| **파인튜닝 지원** | 매우 잘 지원 (가볍고 빠름)                   | Hugging Face와 매우 잘 통합됨                      |
| **사용 용도**     | 데모, 실습용, 기본 생성 테스트에 적합          | 실무 서비스, 연구 실험, 챗봇, Q&A에 적합             |
| **장점**          | 가볍고 빠름, 구조가 단순함                    | 성능 대비 크기 효율 좋고, Colab에서도 잘 동작함       |
| **단점**          | 표현력, 논리성, 길이 제어에서 한계              | 너무 긴 문장에서는 GPT-J 등보다 표현력이 부족할 수 있음 |


In [1]:
# 설치
# uv add transformers datasets  # Hugging Face의 트랜스포머 라이브러리와 데이터셋 도구 설치

In [2]:
from datasets import Dataset

In [3]:
# ===============================
# 필요한 라이브러리 불러오기
# ===============================
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from transformers import DataCollatorForLanguageModeling

import torch

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


In [4]:
from huggingface_hub import login
from dotenv import load_dotenv
import os

# .env 파일 로드
load_dotenv(override=True)

HF_TOKEN = os.getenv("HF_TOKEN")
login(token=HF_TOKEN)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [5]:
# ===============================
# 🔹 사전 학습된 모델 및 토크나이저 불러오기
# ===============================
model_id = "facebook/opt-350m"  # 사용할 사전 학습 언어 모델 ID
tokenizer = AutoTokenizer.from_pretrained(model_id)  # 텍스트를 숫자로 바꾸는 토크나이저 로드
tokenizer.pad_token = tokenizer.eos_token
# OPT 모델은 'pad_token' (빈칸을 채우는 용도)이 기본적으로 정의되어 있지 않습니다.
# 모델이 문장의 길이를 맞춰서 처리할 수 있도록 'pad_token'이 필요합니다.
# 그래서 여기서는 문장의 끝을 나타내는 'eos_token' (end-of-sequence token)을 대신 사용합니다.
# 즉, 빈칸 자리를 eos_token으로 채우도록 설정하는 것입니다.
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype="float32"
)  # Causal Language Modeling용 사전 학습 모델 로드
model.config.pad_token_id = tokenizer.pad_token_id  # 모델 설정에도 pad_token ID를 지정해줘야 에러 방지 가능

Loading weights:   0%|          | 0/388 [00:00<?, ?it/s]

In [6]:
# ===============================
# 🔹 학습할 데이터 구성
# ===============================
data = {
    "text": [
        "### 질문: joy강사의 별명은?\n### 답변: smile",  # [예시] 단순 QA 형태의 문장. 이후 이 형식을 기반으로 추가 학습됨
    ]
}
dataset = Dataset.from_dict(data)  # Hugging Face Dataset 객체로 변환


In [7]:
# ===============================
# 🔹 토큰화 함수 정의 및 적용
# ===============================
def tokenize(example):
    # [포인트] max_length: 64로 고정 → 긴 문장은 자르고, 짧은 문장은 패딩
    return tokenizer(example["text"], padding="max_length", truncation=True, max_length=64)

tokenized_dataset = dataset.map(tokenize)  # 데이터셋 전체에 토큰화 함수 적용

# ===============================
# 🔹 데이터 콜레이터 설정
# ===============================
# 모델에 배치로 넣기 전에 텐서 형태로 묶어주는 역할
# mlm=False → [포인트] 'Causal LM' 방식이므로 MLM(Masked LM)은 사용하지 않음
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

In [8]:
# ===============================
# 🔹 학습 하이퍼파라미터 설정
# ===============================
training_args = TrainingArguments(
    output_dir="./results",  # 학습 결과 저장 폴더
    per_device_train_batch_size=1,  # [포인트] 한 번에 하나씩 학습 → 소규모 실습용 설정
    num_train_epochs=50,  # 전체 데이터셋을 50번 반복 학습
    logging_steps=1,  # 매 스텝마다 로그 출력
    save_strategy="no",  # 학습 중 체크포인트 저장하지 않음
    fp16=True,  # GPU에서 float16 사용 여부 (True로 설정하면 메모리 효율 ↑, CPU에서는 False 유지)
    report_to="none",  # 로그 저장 위치 (None으로 설정 시 WandB 등 외부로 전송 안 함)
)

# ===============================
# 🔹 Trainer 객체 구성 및 학습 시작
# ===============================
trainer = Trainer(
    model=model,  # 학습할 모델
    args=training_args,  # 학습 설정
    train_dataset=tokenized_dataset,  # 학습 데이터셋
    # tokenizer=tokenizer,  # 토크나이저 (로그 기록이나 디코딩 시 사용)
    processing_class=tokenizer,  # 토크나이저 (로그 기록이나 디코딩 시 사용)
    data_collator=data_collator,  # 배치 전처리 콜레이터
)

trainer.train()  # 실제 학습 시작


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
1,4.133230
2,4.000283
3,2.052099
4,0.933938
5,0.304076
6,0.070203
7,0.012850
8,0.008191
9,0.004338
10,0.001161


TrainOutput(global_step=50, training_loss=0.23163017876418962, metrics={'train_runtime': 8.9652, 'train_samples_per_second': 5.577, 'train_steps_per_second': 5.577, 'total_flos': 5824472678400.0, 'train_loss': 0.23163017876418962, 'epoch': 50.0})

In [9]:
# ===============================
# 🔹 학습된 모델로 텍스트 생성 (추론)
# ===============================
input_text = "### 질문: joy강사의 별명은?\n### 답변:"  # [예시] 질문에 대한 답변을 생성해보는 입력
inputs = tokenizer(input_text, return_tensors="pt").to(model.device)  # 토큰화 및 모델에 넣을 수 있도록 텐서로 변환
outputs = model.generate(**inputs, max_new_tokens=50)  # 최대 50 토큰 길이의 응답 생성
print(tokenizer.decode(outputs[0], skip_special_tokens=True))  # 토큰을 사람이 읽을 수 있는 텍스트로 디코딩하여 출력


### 질문: joy강사의 별명은?
### 답변: smiley face
### 답변: smiley smiley smiley smiley smiley smiley smiley smiley smiley smiley smiley smiley smiley smiley smiley smiley smiley smiley smiley


## OPT-350M 파인튜닝- Lora적용

In [10]:
# uv add transformers datasets peft accelerate


In [11]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from transformers import DataCollatorForLanguageModeling
from datasets import Dataset
from peft import get_peft_model, LoraConfig, TaskType
import torch

# 사전 학습 모델 로드
model_id = "facebook/opt-350m"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(model_id)
base_model.config.pad_token_id = tokenizer.pad_token_id
# LoRA 설정 적용
lora_config = LoraConfig(
    r=16,  # 🔹 LoRA의 "랭크(rank)" 값입니다.
          # 학습할 파라미터 수를 줄이는 정도를 설정합니다.
          # r이 작을수록 계산이 가벼워지지만, 너무 작으면 성능이 떨어질 수 있습니다.

    lora_alpha=16,  # 🔹 LoRA가 학습한 정보를 얼마나 강하게 모델에 반영할지 정하는 값입니다.
                    # 일종의 "확대 비율"처럼 작용하며, 일반적으로 r과 함께 조정합니다.

    lora_dropout=0.05,  # 🔹 학습 중 일부 정보를 무작위로 버려 과적합을 막는 기술입니다.
                        # 0.05는 5% 확률로 드롭아웃이 일어나도록 설정한 것입니다.

    bias="none",  # 🔹 기존 모델의 편향(bias) 파라미터는 건드리지 않겠다는 뜻입니다.
                  # 즉, 오직 LoRA 레이어만 학습합니다.

    task_type="CAUSAL_LM"  # 🔹 이 설정이 적용될 작업의 유형입니다.
                           # "CAUSAL_LM"은 일반적인 언어 생성 모델(예: GPT)에서 사용됩니다.
)

model = get_peft_model(base_model, lora_config)


Loading weights:   0%|          | 0/388 [00:00<?, ?it/s]

In [16]:
# 학습할 데이터 구성
data = {
    "text": [
        "### 질문: 우리집 강아지 이름은?\n### 답변:초코송이",
    ]
}
dataset = Dataset.from_dict(data)

# 토큰화
def tokenize(example):
    return tokenizer(example["text"], padding="max_length", truncation=True, max_length=64)

tokenized_dataset = dataset.map(tokenize)

# 데이터 콜레이터
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# 학습 하이퍼파라미터
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=1,
    num_train_epochs=150,
    logging_steps=1,
    save_strategy="no",
    fp16=False,
    report_to="none"
)

# Trainer 구성
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    processing_class=tokenizer,
    data_collator=data_collator
)

trainer.train()

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
1,2.829057
2,2.798043
3,2.814853
4,2.751527
5,2.835640
6,2.790091
7,2.769402
8,2.767240
9,2.603669
10,2.548083


TrainOutput(global_step=150, training_loss=1.4486836206912994, metrics={'train_runtime': 11.3106, 'train_samples_per_second': 13.262, 'train_steps_per_second': 13.262, 'total_flos': 17564015001600.0, 'train_loss': 1.4486836206912994, 'epoch': 150.0})

In [17]:
# 추론
input_text = "### 질문: 우리집 강아지 이름은? \n### 답변:"
inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=100)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

### 질문: 우리집 강아지 이름은? 
### 답변: 이아지 질문: 아지: 아지: 아지: 아지: 아지: 아지: 아지: 아지: 아지: 아지: 아지: 아지:
